***EXPLORATORY DATA ANALYSIS***

In [ ]:
import os
import random
import ast
import re
import gc
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import IPython.display as ipd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.ndimage import gaussian_filter1d
from tqdm.auto import tqdm
import re

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
is_gpu = torch.cuda.is_available()
print(f"Using device: {device}")

base_path = '/kaggle/input/competitions/birdclef-2026'
train_csv_path = os.path.join(base_path, 'train.csv')
soundscapes_csv_path = os.path.join(base_path, 'train_soundscapes_labels.csv')
audio_dir = os.path.join(base_path, 'train_audio')
ss_audio_dir = os.path.join(base_path, 'train_soundscapes')
test_audio_dir = os.path.join(base_path, 'test_soundscapes')

train_df = pd.read_csv(train_csv_path)
soundscapes_df = pd.read_csv(soundscapes_csv_path)

plt.figure(figsize=(14, 6))
top_species = train_df['primary_label'].value_counts().head(30)
sns.barplot(x=top_species.index, y=top_species.values, hue=top_species.index, legend=False, palette='viridis')
plt.xticks(rotation=45)
plt.title('Top 30 Species by Number of Recordings')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

if 'rating' in train_df.columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=train_df, x='rating', hue='rating', legend=False, palette='coolwarm')
    plt.title('Recording Ratings Distribution')
    plt.xlabel('Rating (0 = iNaturalist / Not Available)')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

soundscapes_df['label_count'] = soundscapes_df['primary_label'].apply(lambda x: len(str(x).split(';')))
label_dist = soundscapes_df['label_count'].value_counts().sort_index()

plt.figure(figsize=(8, 4))
sns.barplot(x=label_dist.index, y=label_dist.values, hue=label_dist.index, legend=False, palette='magma')
plt.title('Simultaneous Species in 5-second Soundscape Segments')
plt.xlabel('Number of Species')
plt.ylabel('Segment Count')
plt.tight_layout()
plt.show()

if 'latitude' in train_df.columns and 'longitude' in train_df.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=train_df, x='longitude', y='latitude', alpha=0.3, s=10, color='teal')
    plt.title('Geographic Distribution of Recordings')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.tight_layout()
    plt.show()

print("3 Random Focal Recordings (train_audio)")
focal_samples = train_df.sample(3)
for _, row in focal_samples.iterrows():
    audio_path = os.path.join(audio_dir, row['filename'])
    print(f"Species: {row['primary_label']} | File: {row['filename']}")
    if os.path.exists(audio_path):
        display(ipd.Audio(audio_path))
    else:
        print("File not found.")

print("\n1 Random Soundscape Recording (train_soundscapes)")
if os.path.exists(ss_audio_dir):
    soundscape_files = [f for f in os.listdir(ss_audio_dir) if f.endswith('.ogg')]
    if soundscape_files:
        random_soundscape = random.choice(soundscape_files)
        soundscape_path = os.path.join(ss_audio_dir, random_soundscape)
        print(f"Soundscape File: {random_soundscape}")
        display(ipd.Audio(soundscape_path))
    else:
        print("No soundscape audio files found in the directory.")
else:
    print("Soundscapes directory not found.")

***EXTRACTION OF 5 SECONDS CHUNKS***

In [ ]:
# Librosa is used only for display/analysis here. 
# PyTorch/Torchaudio handles the actual pipeline in later blocks.
import librosa
import librosa.display

def extract_top_chunks(audio_path, sr=32000, duration=5.0, n_mels=256, max_chunks=3):
    """
    Extracts the highest-energy chunks from an audio file.
    Updated n_mels to 256 for higher resolution visualization.
    """
    try:
        y, _ = librosa.load(audio_path, sr=sr)
    except Exception as e:
        print(f"Error loading {audio_path}: {e}")
        return [], []
        
    target_samples = int(sr * duration)
    total_samples = len(y)
    
    # Pad if audio is too short
    if total_samples <= target_samples:
        y_chunk = np.pad(y, (0, target_samples - total_samples))
        mel_spec = librosa.feature.melspectrogram(y=y_chunk, sr=sr, n_mels=n_mels, fmax=16000)
        # Using PCEN logic for visual consistency with our new architecture goals
        mel_spec = librosa.pcen(S=mel_spec * (2**31), sr=sr, hop_length=512, gain=0.98, bias=2, power=0.5, time_constant=0.4)
        return [y_chunk], [mel_spec]
        
    # Break into 5s non-overlapping chunks
    num_full_chunks = total_samples // target_samples
    chunks = []
    energies = []
    
    for i in range(num_full_chunks):
        start = i * target_samples
        end = start + target_samples
        chunk = y[start:end]
        rms = np.mean(librosa.feature.rms(y=chunk))
        chunks.append(chunk)
        energies.append(rms)
        
    # Handle remainder if it's longer than half the target duration
    remainder = total_samples % target_samples
    if remainder > (target_samples // 2):
        chunk = np.pad(y[-remainder:], (0, target_samples - remainder))
        rms = np.mean(librosa.feature.rms(y=chunk))
        chunks.append(chunk)
        energies.append(rms)
        
    # Get indices of the chunks with highest RMS energy
    top_indices = np.argsort(energies)[-max_chunks:][::-1]
    
    best_y = []
    best_mels = []
    for idx in top_indices:
        best_y.append(chunks[idx])
        mel_spec = librosa.feature.melspectrogram(y=chunks[idx], sr=sr, n_mels=n_mels, fmax=16000)
        
        # Apply PCEN for visualization instead of standard DB conversion
        pcen_spec = librosa.pcen(S=mel_spec * (2**31), sr=sr, hop_length=512, gain=0.98, bias=2, power=0.5, time_constant=0.4)
        best_mels.append(pcen_spec)
        
    return best_y, best_mels

print("Displaying High-Resolution PCEN Spectrograms")
sample_files = train_df.sample(2)

for _, row in sample_files.iterrows():
    audio_path = os.path.join(audio_dir, row['filename'])
    species = row['primary_label']
    
    # Check for secondary labels (Soft Targets context)
    secondary = row.get('secondary_labels', '[]')
    
    if os.path.exists(audio_path):
        print(f"\nPrimary Species: {species} | Secondary: {secondary} | File: {row['filename']}")
        
        # Using n_mels=256 as planned for the new architecture
        y_chunks, pcen_specs = extract_top_chunks(audio_path, n_mels=256, max_chunks=3)
        
        for i, (y_chunk, pcen_spec) in enumerate(zip(y_chunks, pcen_specs)):
            plt.figure(figsize=(6, 2))
            # Display PCEN representation
            librosa.display.specshow(pcen_spec, sr=32000, x_axis='time', y_axis='mel', fmax=16000)
            plt.colorbar(format='%+2.0f')
            plt.title(f'PCEN Chunk {i+1} - {species}')
            plt.tight_layout()
            plt.show()
            
            display(ipd.Audio(y_chunk, rate=32000))
    else:
        print(f"File {audio_path} not found.")

***DATA PREPROCESSING AND EXTRACTION***

In [ ]:
# Handle singletons to prevent train_test_split crash
counts = train_df['primary_label'].value_counts()
singletons = counts[counts == 1].index
if len(singletons) > 0:
    print(f"Found {len(singletons)} singleton species. Duplicating records...")
    df_singletons = train_df[train_df['primary_label'].isin(singletons)]
    train_df = pd.concat([train_df, df_singletons], ignore_index=True)

# Encode primary labels
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['primary_label'])
num_classes = len(label_encoder.classes_)

# Parse secondary labels from string representation to Python lists
train_df['secondary_labels_list'] = train_df['secondary_labels'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

# Cap samples at 150 per class to prevent heavy class imbalance
df_balanced = train_df.groupby('primary_label').head(150).reset_index(drop=True)

class BirdWaveformDataset(Dataset):
    def __init__(self, df, audio_dir, label_encoder, target_sr=32000, duration=5.0, is_train=True):
        self.df = df
        self.audio_dir = audio_dir
        self.label_encoder = label_encoder
        self.num_classes = len(label_encoder.classes_)
        self.target_sr = target_sr
        self.target_samples = int(target_sr * duration)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.audio_dir, row['filename'])
        
        waveform, sr = torchaudio.load(file_path)
        
        # Convert multi-channel audio to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # Resample audio if the sample rate differs from the target sample rate
        if sr != self.target_sr:
            resampler = T.Resample(sr, self.target_sr)
            waveform = resampler(waveform)
            
        # Apply dynamic time shifting or cropping during training
        if waveform.shape[1] > self.target_samples:
            if self.is_train:
                max_start = waveform.shape[1] - self.target_samples
                start = torch.randint(0, max_start, (1,)).item()
            else:
                start = 0
            waveform = waveform[:, start:start + self.target_samples]
            
        elif waveform.shape[1] < self.target_samples:
            pad_amount = self.target_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_amount))
            
        # Waveform augmentation: inject Gaussian white noise randomly during training
        if self.is_train and torch.rand(1).item() < 0.5:
            noise_amplitude = 0.005 * torch.rand(1).item()
            waveform += torch.randn_like(waveform) * noise_amplitude
            
        # Soft targets implementation
        target = torch.zeros(self.num_classes, dtype=torch.float32)
        
        # Primary species gets full confidence (1.0)
        target[row['label_encoded']] = 1.0
        
        # Secondary species get partial confidence (0.3) to teach co-occurrence
        for sec_species in row['secondary_labels_list']:
            if sec_species in self.label_encoder.classes_:
                sec_idx = self.label_encoder.transform([sec_species])[0]
                target[sec_idx] = 0.3
                
        return waveform, target

# Split maintaining class stratification based on the primary label
train_df_split, val_df_split = train_test_split(
    df_balanced, test_size=0.2, stratify=df_balanced['label_encoded'], random_state=42
)

train_dataset = BirdWaveformDataset(train_df_split, audio_dir, label_encoder, is_train=True)
val_dataset = BirdWaveformDataset(val_df_split, audio_dir, label_encoder, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

print("Datasets ready")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

***MODEL DEFINITION***

In [ ]:
# PCEN, SpecAugment, Asymmetric CNN, and Dropout implementation
class AudioToSpectrogramGPU(nn.Module):
    def __init__(self, sr=32000, n_mels=256, n_fft=2048, hop_length=512, f_min=40, f_max=15000):
        super(AudioToSpectrogramGPU, self).__init__()
        
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            f_min=f_min,
            f_max=f_max,
            n_mels=n_mels,
            power=2.0 
        )
        
        # PCEN hyperparameters
        self.eps = 1e-6
        self.s = 0.025
        self.alpha = 0.98
        self.delta = 2.0
        self.r = 0.5
        
        # SpecAugment masks
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=36)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=64)

    def forward(self, waveform):
        x = self.mel_spec(waveform)
        
        ema = x.clone()
        for t in range(1, x.size(-1)):
            ema[..., t] = (1 - self.s) * ema[..., t - 1] + self.s * x[..., t]
            
        x = (x / (self.eps + ema)**self.alpha + self.delta)**self.r - self.delta**self.r
        x = (x - x.mean()) / (x.std() + 1e-6)
        
        if self.training:
            x = self.freq_mask(x)
            x = self.time_mask(x)
            
        return x

class AsymmetricConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool_freq=2, pool_time=1):
        super(AsymmetricConvBlock, self).__init__()
        self.conv_f = nn.Conv2d(in_channels, out_channels, kernel_size=(5, 1), padding=(2, 0), bias=False)
        self.bn_f = nn.BatchNorm2d(out_channels)
        self.conv_t = nn.Conv2d(out_channels, out_channels, kernel_size=(1, 5), padding=(0, 2), bias=False)
        self.bn_t = nn.BatchNorm2d(out_channels)
        self.pool = nn.MaxPool2d(kernel_size=(pool_freq, pool_time))

    def forward(self, x):
        return self.pool(F.relu(self.bn_t(self.conv_t(F.relu(self.bn_f(self.conv_f(x)))))))

class AttentivePooling(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(AttentivePooling, self).__init__()
        self.attention = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)
        self.classifier = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)

    def forward(self, x):
        att_weights = torch.softmax(self.attention(x), dim=-1)
        frame_logits = self.classifier(x)
        clip_logits = torch.sum(att_weights * frame_logits, dim=-1)
        return clip_logits, frame_logits

class BirdSED_AsymmetricCNN(nn.Module):
    def __init__(self, num_classes):
        super(BirdSED_AsymmetricCNN, self).__init__()
        
        self.audio_extractor = AudioToSpectrogramGPU()
        
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        
        self.block1 = AsymmetricConvBlock(32, 64, pool_freq=2, pool_time=1)
        self.block2 = AsymmetricConvBlock(64, 128, pool_freq=2, pool_time=2)
        self.block3 = AsymmetricConvBlock(128, 256, pool_freq=2, pool_time=2)
        self.block4 = AsymmetricConvBlock(256, 512, pool_freq=2, pool_time=2)
        
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.dropout = nn.Dropout(0.5)
        self.sed_head = AttentivePooling(in_channels=512, num_classes=num_classes)

    def forward(self, waveform):
        x = self.audio_extractor(waveform)
        
        x = self.stem(x) # Compress spatial dimensions quickly from 1 to 32 channels
        x = self.block4(self.block3(self.block2(self.block1(x))))
        
        x = self.dropout(self.freq_pool(x).squeeze(2))
        clip_logits, _ = self.sed_head(x)
        return clip_logits

model = BirdSED_AsymmetricCNN(num_classes=num_classes).to(device)

dummy_waveform = torch.randn(1, 1, 32000 * 5).to(device)

model.eval()
with torch.no_grad():
    logits = model(dummy_waveform)

print("Architecture test")
print(f"Input waveform shape: {dummy_waveform.shape}")
print(f"Output clip predictions shape: {logits.shape}")

***FOCAL AUDIO TRAINING***

In [ ]:
# Custom Focal Loss for multi-label classification to handle class imbalance dynamically
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss) 
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

# Mixup adaptation for soft targets
def apply_mixup(waveforms, labels):
    indices = torch.randperm(waveforms.size(0)).to(waveforms.device)
    shuffled_waveforms = waveforms[indices]
    shuffled_labels = labels[indices]
    
    mixed_waveforms = (waveforms + shuffled_waveforms) / 2.0
    mixed_labels = torch.max(labels, shuffled_labels)
    
    return mixed_waveforms, mixed_labels

save_path_A = 'sed_focal_phaseA_Asymmetric.pth'
best_val_loss = float('inf')

if os.path.exists(save_path_A):
    model.load_state_dict(torch.load(save_path_A, map_location=device))
    print("Recovered previous weights successfully.")

criterion = FocalLoss(alpha=1.0, gamma=2.0)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 30 if is_gpu else 2
THRESHOLD = 0.3 

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=3e-3, 
    steps_per_epoch=len(train_loader), 
    epochs=epochs,
    pct_start=0.2, 
    div_factor=10.0
)

print("Starting Phase A training")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    train_bar = tqdm(train_loader, desc=f"Train [Epoch {epoch+1}/{epochs}]")
    for waveforms, labels in train_bar:
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        
        waveforms, labels = apply_mixup(waveforms, labels)
        
        optimizer.zero_grad()
        logits = model(waveforms)
        loss = criterion(logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step() 
        
        train_loss += loss.item()
        
        current_lr = scheduler.get_last_lr()[0]
        train_bar.set_postfix(FLoss=f"{loss.item():.4f}", LR=f"{current_lr:.5f}")
        
        del waveforms, labels, logits, loss
        if not is_gpu: break

    model.eval()
    val_loss = 0.0
    all_preds, all_true, all_probs = [], [], []
    
    val_bar = tqdm(val_loader, desc=f"Val   [Epoch {epoch+1}/{epochs}]")
    with torch.no_grad():
        for waveforms, labels in val_bar:
            waveforms = waveforms.to(device)
            labels = labels.to(device)
            
            logits = model(waveforms)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > THRESHOLD).int()
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            
            binary_true = (labels > 0.5).int()
            all_true.extend(binary_true.cpu().numpy())
            
            del waveforms, labels, logits, loss, probs, preds
            if not is_gpu: break

    if is_gpu:
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Focal Loss: {train_loss/len(train_loader):.4f} | Val Focal Loss: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path_A)
            print(f"New best model saved with validation focal loss: {best_val_loss:.4f}\n")
        else:
            print()
            
    gc.collect()
    if is_gpu:
        torch.cuda.empty_cache()

***SOUNDSCAPES TRAINING***

In [ ]:
def time_to_seconds(t_str):
    h, m, s = map(int, str(t_str).split(':'))
    return h * 3600 + m * 60 + s

def parse_labels(val):
    if pd.isna(val): return []
    return [str(l) for l in re.split(r'[;,\s]+', str(val).strip()) if l]

class SoundscapeWaveformDataset(Dataset):
    def __init__(self, csv_path, audio_dir, label_encoder, target_sr=32000, duration=5.0):
        df = pd.read_csv(csv_path)
        df['start_sec'] = df['start'].apply(time_to_seconds)
        df['primary_label'] = df['primary_label'].apply(parse_labels)
        self.df = df
        self.audio_dir = audio_dir
        self.label_encoder = label_encoder
        self.num_classes = len(label_encoder.classes_)
        self.target_sr = target_sr
        self.frames_per_chunk = int(target_sr * duration)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.audio_dir, row['filename'])
        
        frame_offset = int(row['start_sec'] * self.target_sr)
        
        try:
            waveform, sr = torchaudio.load(file_path, frame_offset=frame_offset, num_frames=self.frames_per_chunk)
        except Exception:
            waveform = torch.zeros((1, self.frames_per_chunk))
            sr = self.target_sr
            
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        if waveform.shape[1] < self.frames_per_chunk:
            waveform = torch.nn.functional.pad(waveform, (0, self.frames_per_chunk - waveform.shape[1]))

        target = torch.zeros(self.num_classes, dtype=torch.float32)
        for species in row['primary_label']:
            try:
                class_idx = self.label_encoder.transform([species])[0]
                target[class_idx] = 1.0
            except ValueError:
                pass
                
        return waveform, target

print("Preparing Phase B datasets")
ss_dataset_full = SoundscapeWaveformDataset(soundscapes_csv_path, ss_audio_dir, label_encoder)

train_size = int(0.8 * len(ss_dataset_full))
val_size = len(ss_dataset_full) - train_size
train_ss, val_ss = torch.utils.data.random_split(ss_dataset_full, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_ss_loader = DataLoader(train_ss, batch_size=32, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_ss_loader = DataLoader(val_ss, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

# Phase B training (fine-tuning)
print("Starting Phase B soundscapes fine-tuning")

try:
    model.load_state_dict(torch.load('sed_focal_phaseA_Asymmetric.pth', map_location=device))
    print("Loaded Phase A Asymmetric weights successfully.")
except Exception as e:
    print(f"Phase A weights not found or error loading: {e}")

criterion_b = FocalLoss(alpha=1.0, gamma=2.0) 
optimizer_b = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

epochs_b = 15 if is_gpu else 1
best_val_loss_b = float('inf')
save_path_B = 'sed_final_phaseB_Asymmetric.pth'
THRESHOLD = 0.3

scheduler_b = optim.lr_scheduler.OneCycleLR(
    optimizer_b, 
    max_lr=5e-4, 
    steps_per_epoch=len(train_ss_loader), 
    epochs=epochs_b,
    pct_start=0.2, 
    div_factor=10.0
)

for epoch in range(epochs_b):
    model.train()
    train_loss = 0.0
    
    train_bar = tqdm(train_ss_loader, desc=f"PB Train [Ep {epoch+1}/{epochs_b}]")
    for waveforms, labels in train_bar:
        waveforms, labels = waveforms.to(device), labels.to(device)
        
        optimizer_b.zero_grad()
        logits = model(waveforms)
        loss = criterion_b(logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer_b.step()
        scheduler_b.step()
        
        train_loss += loss.item()
        
        current_lr = scheduler_b.get_last_lr()[0]
        train_bar.set_postfix(FLoss=f"{loss.item():.4f}", LR=f"{current_lr:.5f}")
        
        del waveforms, labels, logits, loss
        if not is_gpu: break
        
    model.eval()
    val_loss = 0.0
    all_preds, all_true, all_probs = [], [], []
    
    val_bar = tqdm(val_ss_loader, desc=f"PB Val   [Ep {epoch+1}/{epochs_b}]")
    with torch.no_grad():
        for waveforms, labels in val_bar:
            waveforms, labels = waveforms.to(device), labels.to(device)
            logits = model(waveforms)
            loss = criterion_b(logits, labels)
            val_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > THRESHOLD).int()
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            
            del waveforms, labels, logits, loss, probs, preds
            if not is_gpu: break

    if is_gpu:
        avg_val_loss = val_loss / len(val_ss_loader)
        
        print(f"Phase B Epoch [{epoch+1}/{epochs_b}] | Train Focal Loss: {train_loss/len(train_ss_loader):.4f} | Val Focal Loss: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss_b:
            best_val_loss_b = avg_val_loss
            torch.save(model.state_dict(), save_path_B)
            print(f"New best fine-tuned model saved with validation focal loss: {best_val_loss_b:.4f}\n")
        else:
            print()
            
    gc.collect()
    if is_gpu:
        torch.cuda.empty_cache()